# YOLOv8 Card Detector Training

This notebook trains a **YOLOv8-nano** model to detect playing cards in images (Stage 1 of the 24 Game Solver pipeline).

- **Input:** raw photos containing playing cards
- **Output:** bounding boxes around each card (single class: `card`)
- **Export:** TFLite FP16 at 320×320 for on-device inference in Flutter

> **Runtime:** Set to GPU (Runtime → Change runtime type → T4 GPU) before running.

## Setup

Install dependencies and import libraries.

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow

import os
import glob
from pathlib import Path
from ultralytics import YOLO

## Data

Download a playing card detection dataset from [Roboflow Universe](https://universe.roboflow.com/).

**Before running this cell:**
1. Create a free account at https://app.roboflow.com/
2. Search Roboflow Universe for `playing cards` and pick a dataset with bounding box annotations
3. Replace the placeholder values below with your API key and chosen dataset details

Alternatively, download the dataset manually and upload it to Colab under `data/cards_detection/`.

In [ ]:
# Download a playing card detection dataset from Roboflow
# Search Roboflow Universe for "playing cards" → pick one with bounding box annotations
#
# Replace ROBOFLOW_API_KEY and dataset details with actual values after selecting a dataset.
# Alternatively, download manually and upload to Colab.

from roboflow import Roboflow

# TODO: Replace with your actual Roboflow API key and dataset details
# You can get a free API key at https://app.roboflow.com/
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- replace before running
WORKSPACE = "YOUR_WORKSPACE"       # <-- replace before running
PROJECT = "YOUR_PROJECT"           # <-- replace before running
VERSION = 1                        # <-- replace before running

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8", location="data/cards_detection")

print(f"Dataset downloaded to: data/cards_detection")
print(f"Train images: {len(os.listdir('data/cards_detection/train/images'))}")
print(f"Val images: {len(os.listdir('data/cards_detection/valid/images'))}")

### Remap Classes

The dataset may have fine-grained labels (e.g. `ace_of_spades`, `2_of_hearts`, …). We collapse all 52 card classes into a single class `card` (class ID 0) since Stage 1 only needs to *locate* cards — Stage 2 will classify the rank/suit.

In [ ]:
# Remap all class labels to single class 0 ("card")
# YOLO format: class_id center_x center_y width height (normalized)

def remap_labels_to_single_class(labels_dir: str) -> int:
    """Rewrite all YOLO label files to use class 0 for every object."""
    count = 0
    for label_file in glob.glob(os.path.join(labels_dir, "*.txt")):
        with open(label_file, "r") as f:
            lines = f.readlines()

        remapped = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                # Replace class ID (first field) with 0
                parts[0] = "0"
                remapped.append(" ".join(parts))

        with open(label_file, "w") as f:
            f.write("\n".join(remapped) + "\n" if remapped else "")
        count += 1
    return count

train_count = remap_labels_to_single_class("data/cards_detection/train/labels")
val_count = remap_labels_to_single_class("data/cards_detection/valid/labels")
print(f"Remapped {train_count} train + {val_count} val label files to single class 'card'")

### Create data.yaml

Write the YOLO dataset config file that points to the train/val splits and defines the single `card` class.

In [ ]:
# Create YOLO data config
data_yaml = """
path: /content/data/cards_detection
train: train/images
val: valid/images

nc: 1
names:
  0: card
"""

with open("data/cards_detection/data.yaml", "w") as f:
    f.write(data_yaml.strip())

print("Created data.yaml")
print(open("data/cards_detection/data.yaml").read())

## Training

Train **YOLOv8-nano** for up to 100 epochs with early stopping (patience=15). Training at 640×640 gives better feature quality before we export at 320×320 for inference.

Expected training time on a T4 GPU: ~20–40 minutes depending on dataset size.

In [ ]:
# Train YOLOv8-nano
model = YOLO("yolov8n.pt")

results = model.train(
    data="data/cards_detection/data.yaml",
    epochs=100,
    imgsz=640,       # train at 640 for better features
    batch=16,
    patience=15,     # early stopping
    project="outputs",
    name="yolo_card_detector",
    exist_ok=True,
)

print("Training complete!")
print(f"Best model: outputs/yolo_card_detector/weights/best.pt")

## Evaluation

Evaluate the best checkpoint on the validation set. We target **mAP@0.5 > 0.90** for reliable card detection in the Flutter app.

In [ ]:
# Evaluate on validation set
best_model = YOLO("outputs/yolo_card_detector/weights/best.pt")
metrics = best_model.val(data="data/cards_detection/data.yaml")

print(f"mAP@0.5:     {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Target: mAP@0.5 > 0.90
assert metrics.box.map50 > 0.85, f"mAP@0.5 too low: {metrics.box.map50:.4f}"

### Visualize Predictions

Run inference on a sample of validation images and display the annotated results.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Run predictions on sample validation images
val_images = glob.glob("data/cards_detection/valid/images/*.jpg")[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, val_images):
    results = best_model.predict(img_path, conf=0.5, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[:, :, ::-1])  # BGR to RGB
    ax.axis("off")
    ax.set_title(Path(img_path).name)

plt.suptitle("YOLOv8-nano Card Detector — Validation Predictions", fontsize=16)
plt.tight_layout()
plt.savefig("outputs/yolo_card_detector/val_predictions.png", dpi=150)
plt.show()

## Export

Export the trained model to **TFLite FP16** at 320×320 resolution for on-device inference.

After exporting, copy the `.tflite` file to `flutter_app/assets/models/card_detector.tflite`.

In [ ]:
# Export to TFLite at inference size 320x320
# FP16 quantization for good accuracy + small size
exported_path = best_model.export(
    format="tflite",
    imgsz=320,
    int8=False,
    half=True,  # FP16
)

import os
tflite_files = glob.glob("outputs/yolo_card_detector/weights/*float16.tflite") + \
               glob.glob("outputs/yolo_card_detector/weights/*.tflite")
for f in tflite_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"{f}: {size_mb:.1f} MB")

print("\nCopy the .tflite file to flutter_app/assets/models/card_detector.tflite")